# Economic Results Visualization - Objective 2

This notebook visualizes the economic results comparing Scenario 0 (BAU), Scenario 1 (Policy Mandate), and Scenario 2 (Book & Claim solution).

**Key Insights:**
- Feedstock Wall impact (HEFA supply cap)
- Logistics penalty cost in S1
- S2 savings opportunity


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded")


## 1. Data Loading


In [ ]:
# Load the economic results
df = pd.read_csv('data/metrics/obj2_economic_results.csv')

print(f"✅ Data loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"\nYear range: {df['Year'].min()} - {df['Year'].max()}")
print("\nFirst 5 rows:")
df.head()


## 2. Feature Engineering: Scenario 2 (Book & Claim)


In [ ]:
# S2 = S1 without logistics penalty (Book & Claim eliminates physical trucking)
df['S2_Total_Cost_Bn'] = df['S1_Total_Cost_Bn'] - df['S1_Logistics_Penalty_Bn']

print("✅ Scenario 2 cost calculated")
print(f"\nS2 Total Cost (cumulative 2026-2050): ${df['S2_Total_Cost_Bn'].sum():.2f} Billion")
print(f"S1 Total Cost (cumulative 2026-2050): ${df['S1_Total_Cost_Bn'].sum():.2f} Billion")
print(f"Savings from S2: ${df['S1_Logistics_Penalty_Bn'].sum():.2f} Billion")


## 3. Visualization 1: The Feedstock Wall


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Plot SAF demand
ax.plot(df['Year'], df['S1_SAF_Vol_Mt'], 
        color='#2E86AB', linewidth=2.5, label='S1 SAF Demand', marker='o', markersize=4)

# Plot HEFA supply cap (the wall)
ax.axhline(y=df['HEFA_EU_Supply_Cap_Mt'].iloc[0], 
           color='#A23B72', linestyle='--', linewidth=2.5, 
           label=f'HEFA Supply Cap ({df["HEFA_EU_Supply_Cap_Mt"].iloc[0]} Mt)')

# Find intersection point (wall hit)
wall_hit_years = df[df['Wall_Hit'] == True]['Year']
if len(wall_hit_years) > 0:
    first_wall_hit = wall_hit_years.min()
    wall_hit_idx = df[df['Year'] == first_wall_hit].index[0]
    wall_hit_demand = df.loc[wall_hit_idx, 'S1_SAF_Vol_Mt']
    
    # Add annotation at crash point
    ax.annotate(f'Wall Hit: {int(first_wall_hit)}', 
                xy=(first_wall_hit, wall_hit_demand),
                xytext=(first_wall_hit + 2, wall_hit_demand + 1),
                arrowprops=dict(arrowstyle='->', color='#A23B72', lw=1.5),
                fontsize=11, fontweight='bold', color='#A23B72',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#A23B72', alpha=0.8))

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('SAF Volume (Mt)', fontsize=12, fontweight='bold')
ax.set_title('The Feedstock Wall: SAF Demand vs HEFA Supply Cap', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"📊 Wall first hit in: {int(wall_hit_years.min()) if len(wall_hit_years) > 0 else 'Never'}")


## 4. Visualization 2: The Green Premium & S2 Savings (Stacked Bar Chart)


In [ ]:
# Calculate cumulative totals for 2050 snapshot
cumulative_s0 = df['S0_Total_Cost_Bn'].sum()
cumulative_s1 = df['S1_Total_Cost_Bn'].sum()
cumulative_s2 = df['S2_Total_Cost_Bn'].sum()
cumulative_logistics = df['S1_Logistics_Penalty_Bn'].sum()

# Break down S1 into components (excluding logistics for base)
s1_base = cumulative_s1 - cumulative_logistics
s1_carbon = df['S1_Carbon_Cost_Bn'].sum()
s1_fuel = df['S1_Jet_Cost_Bn'].sum() + df['S1_SAF_Cost_Bn'].sum()

# Break down S0
s0_fuel = df['S0_Fuel_Cost_Bn'].sum()
s0_carbon = df['S0_Carbon_Cost_Bn'].sum()

# Break down S2 (same as S1 but without logistics)
s2_fuel = s1_fuel
s2_carbon = s1_carbon

# Prepare data for stacked bar chart
scenarios = ['Scenario 0\n(BAU)', 'Scenario 1\n(Policy)', 'Scenario 2\n(Book & Claim)']
fuel_costs = [s0_fuel, s1_fuel, s2_fuel]
carbon_costs = [s0_carbon, s1_carbon, s2_carbon]
logistics_costs = [0, cumulative_logistics, 0]  # Only S1 has logistics

x = np.arange(len(scenarios))
width = 0.6

fig, ax = plt.subplots(figsize=(11, 7))

# Create stacked bars
p1 = ax.bar(x, fuel_costs, width, label='Fuel Cost', color='#2E86AB', alpha=0.8)
p2 = ax.bar(x, carbon_costs, width, bottom=fuel_costs, label='Carbon Cost', color='#F18F01', alpha=0.8)
p3 = ax.bar(x, logistics_costs, width, bottom=[fuel_costs[i] + carbon_costs[i] for i in range(3)], 
            label='Logistics Penalty (Waste)', color='#C73E1D', alpha=0.9)

# Add value labels on top of bars
for i, (scenario, total) in enumerate(zip(scenarios, [cumulative_s0, cumulative_s1, cumulative_s2])):
    ax.text(i, total + 10, f'${total:.0f}B', 
            ha='center', va='bottom', fontweight='bold', fontsize=11)

# Highlight S2 savings
ax.text(2, cumulative_s2 + 30, f'Savings:\n${cumulative_logistics:.0f}B', 
        ha='center', va='bottom', fontweight='bold', fontsize=10, 
        color='#06A77D', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#06A77D', alpha=0.8))

ax.set_ylabel('Cumulative Cost (Billion USD)', fontsize=12, fontweight='bold')
ax.set_title('Cumulative Total Cost Comparison (2026-2050)\nGreen Premium & S2 Savings', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.show()

print(f"💰 S0 Total: ${cumulative_s0:.2f}B")
print(f"💰 S1 Total: ${cumulative_s1:.2f}B (includes ${cumulative_logistics:.2f}B logistics penalty)")
print(f"💰 S2 Total: ${cumulative_s2:.2f}B (saves ${cumulative_logistics:.2f}B)")


In [ ]:
# Calculate cumulative sums
df['S0_Cumulative_Cost'] = df['S0_Total_Cost_Bn'].cumsum()
df['S1_Cumulative_Cost'] = df['S1_Total_Cost_Bn'].cumsum()
df['S2_Cumulative_Cost'] = df['S2_Total_Cost_Bn'].cumsum()

fig, ax = plt.subplots(figsize=(13, 7))

# Plot cumulative costs
ax.plot(df['Year'], df['S0_Cumulative_Cost'], 
        color='#6C757D', linewidth=2.5, label='Scenario 0 (BAU)', marker='o', markersize=5, alpha=0.8)
ax.plot(df['Year'], df['S1_Cumulative_Cost'], 
        color='#A23B72', linewidth=2.5, label='Scenario 1 (Policy)', marker='s', markersize=5, alpha=0.8)
ax.plot(df['Year'], df['S2_Cumulative_Cost'], 
        color='#06A77D', linewidth=2.5, label='Scenario 2 (Book & Claim)', marker='^', markersize=5, alpha=0.8)

# Fill area between S1 and S2 to show savings
ax.fill_between(df['Year'], df['S1_Cumulative_Cost'], df['S2_Cumulative_Cost'], 
                 alpha=0.3, color='#06A77D', label='S2 Savings Area')

# Add annotation for final savings
final_savings = df['S1_Cumulative_Cost'].iloc[-1] - df['S2_Cumulative_Cost'].iloc[-1]
ax.annotate(f'Total S2 Savings:\n${final_savings:.0f}B', 
            xy=(df['Year'].iloc[-1], df['S2_Cumulative_Cost'].iloc[-1]),
            xytext=(df['Year'].iloc[-1] - 3, df['S2_Cumulative_Cost'].iloc[-1] - 100),
            arrowprops=dict(arrowstyle='->', color='#06A77D', lw=1.5),
            fontsize=11, fontweight='bold', color='#06A77D',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#06A77D', alpha=0.9))

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Cumulative Cost (Billion USD)', fontsize=12, fontweight='bold')
ax.set_title('Cumulative Total Cost Over Time: Diverging Paths of S0, S1, and S2', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"📈 Final cumulative costs (2050):")
print(f"   S0: ${df['S0_Cumulative_Cost'].iloc[-1]:.2f}B")
print(f"   S1: ${df['S1_Cumulative_Cost'].iloc[-1]:.2f}B")
print(f"   S2: ${df['S2_Cumulative_Cost'].iloc[-1]:.2f}B")
print(f"   S2 Savings: ${final_savings:.2f}B")
